# Professional Practice X3: Unsupervised Preprocessing

## Data Cleaning Pipeline for Unsupervised Learning

**Production Challenge**: Real-world data is messy - outliers, different scales, missing values. How do you prepare data for clustering?

**Goal**: Build a complete preprocessing pipeline: outlier detection → scaling → feature engineering

**Use Case**: Data preparation before clustering, dimensionality reduction, or anomaly detection

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.datasets import make_blobs
from sklearn.decomposition import PCA

np.random.seed(42)
print('✅ Loaded!')

## Step 1: Generate Messy Data

Real data has outliers, different scales, and noise.

In [ ]:
# Create realistic messy data
# Normal data
X_normal, _ = make_blobs(n_samples=500, centers=3, cluster_std=1.0, random_state=42)

# Add outliers (10%)
n_outliers = 50
X_outliers = np.random.uniform(low=-15, high=15, size=(n_outliers, 2))

# Combine
X_raw = np.vstack([X_normal, X_outliers])

# Add different scales to features
X_raw[:, 0] = X_raw[:, 0] * 10  # Feature 1: large scale
X_raw[:, 1] = X_raw[:, 1] * 0.1  # Feature 2: small scale

print(f"Dataset: {X_raw.shape}")
print(f"Normal samples: {len(X_normal)}")
print(f"Outliers: {n_outliers}")
print(f"\nFeature statistics:")
print(f"  Feature 1 - Range: [{X_raw[:, 0].min():.1f}, {X_raw[:, 0].max():.1f}], Std: {X_raw[:, 0].std():.1f}")
print(f"  Feature 2 - Range: [{X_raw[:, 1].min():.1f}, {X_raw[:, 1].max():.1f}], Std: {X_raw[:, 1].std():.1f}")

# Visualize
plt.figure(figsize=(10, 6))
plt.scatter(X_normal[:, 0], X_normal[:, 1], s=30, alpha=0.6, label='Normal', c='blue')
plt.scatter(X_outliers[:, 0], X_outliers[:, 1], s=100, alpha=0.8, label='Outliers', 
           c='red', marker='x', linewidths=2)
plt.xlabel('Feature 1 (large scale)', fontsize=12)
plt.ylabel('Feature 2 (small scale)', fontsize=12)
plt.title('Raw Data with Outliers', fontweight='bold', fontsize=13)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('✅ Messy data generated!')

## Step 2: Outlier Detection

Compare Isolation Forest and LOF for outlier detection.

In [ ]:
# Method 1: Isolation Forest
iso = IsolationForest(contamination=0.1, random_state=42)
outliers_iso = iso.fit_predict(X_raw)
scores_iso = iso.score_samples(X_raw)

# Method 2: Local Outlier Factor
lof = LocalOutlierFactor(contamination=0.1)
outliers_lof = lof.fit_predict(X_raw)
scores_lof = lof.negative_outlier_factor_

print("=== OUTLIER DETECTION ===\n")
print(f"Isolation Forest:")
print(f"  Detected outliers: {sum(outliers_iso == -1)}")
print(f"  True outliers captured: {sum(outliers_iso[-n_outliers:] == -1)}/{n_outliers}")

print(f"\nLOF:")
print(f"  Detected outliers: {sum(outliers_lof == -1)}")
print(f"  True outliers captured: {sum(outliers_lof[-n_outliers:] == -1)}/{n_outliers}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (name, labels, scores) in zip(axes, [
    ('Isolation Forest', outliers_iso, scores_iso),
    ('LOF', outliers_lof, scores_lof)
]):
    # Plot normal vs outliers
    normal_mask = labels == 1
    outlier_mask = labels == -1
    
    scatter = ax.scatter(X_raw[normal_mask, 0], X_raw[normal_mask, 1], 
                        c=scores[normal_mask], cmap='Blues', s=30, alpha=0.6, 
                        label='Normal', edgecolors='black', linewidths=0.3)
    ax.scatter(X_raw[outlier_mask, 0], X_raw[outlier_mask, 1], 
              c='red', s=100, marker='x', linewidths=2, label='Outlier')
    
    ax.set_xlabel('Feature 1', fontsize=11)
    ax.set_ylabel('Feature 2', fontsize=11)
    ax.set_title(f'{name} Detection', fontweight='bold', fontsize=12)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=ax, label='Anomaly Score')

plt.tight_layout()
plt.show()
print('\n✅ Outlier detection complete!')

## Step 3: Remove Outliers

Clean the dataset using Isolation Forest results.

In [ ]:
# Remove outliers
X_clean = X_raw[outliers_iso == 1]

print(f"Before: {len(X_raw)} samples")
print(f"After: {len(X_clean)} samples")
print(f"Removed: {len(X_raw) - len(X_clean)} samples ({(len(X_raw) - len(X_clean))/len(X_raw)*100:.1f}%)")

# Visualize cleaned data
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(X_raw[:, 0], X_raw[:, 1], s=30, alpha=0.6, c='blue')
axes[0].scatter(X_raw[outliers_iso == -1, 0], X_raw[outliers_iso == -1, 1], 
               s=100, c='red', marker='x', linewidths=2)
axes[0].set_xlabel('Feature 1', fontsize=11)
axes[0].set_ylabel('Feature 2', fontsize=11)
axes[0].set_title('Before: With Outliers', fontweight='bold', fontsize=12)
axes[0].grid(True, alpha=0.3)

axes[1].scatter(X_clean[:, 0], X_clean[:, 1], s=30, alpha=0.6, c='green')
axes[1].set_xlabel('Feature 1', fontsize=11)
axes[1].set_ylabel('Feature 2', fontsize=11)
axes[1].set_title('After: Outliers Removed', fontweight='bold', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('✅ Outliers removed!')

## Step 4: Compare Scaling Methods

Different scalers for different scenarios.

In [ ]:
# Apply different scalers
scalers = {
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler()
}

scaled_data = {}
for name, scaler in scalers.items():
    scaled_data[name] = scaler.fit_transform(X_clean)

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Original
ax = axes[0, 0]
ax.scatter(X_clean[:, 0], X_clean[:, 1], s=30, alpha=0.6, c='blue')
ax.set_xlabel('Feature 1', fontsize=11)
ax.set_ylabel('Feature 2', fontsize=11)
ax.set_title('Original (Unscaled)', fontweight='bold', fontsize=12)
ax.grid(True, alpha=0.3)
ax.text(0.02, 0.98, f'F1 range: [{X_clean[:, 0].min():.1f}, {X_clean[:, 0].max():.1f}]\nF2 range: [{X_clean[:, 1].min():.2f}, {X_clean[:, 1].max():.2f}]',
       transform=ax.transAxes, verticalalignment='top', fontsize=9,
       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Scaled versions
for idx, (name, X_scaled) in enumerate(scaled_data.items(), 1):
    ax = axes.flatten()[idx]
    ax.scatter(X_scaled[:, 0], X_scaled[:, 1], s=30, alpha=0.6, c='green')
    ax.set_xlabel('Feature 1 (scaled)', fontsize=11)
    ax.set_ylabel('Feature 2 (scaled)', fontsize=11)
    ax.set_title(name, fontweight='bold', fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')
    ax.text(0.02, 0.98, f'F1: [{X_scaled[:, 0].min():.2f}, {X_scaled[:, 0].max():.2f}]\nF2: [{X_scaled[:, 1].min():.2f}, {X_scaled[:, 1].max():.2f}]',
           transform=ax.transAxes, verticalalignment='top', fontsize=9,
           bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

# Print statistics
print("=== SCALING COMPARISON ===\n")
for name, X_scaled in scaled_data.items():
    print(f"{name}:")
    print(f"  Feature 1 - Mean: {X_scaled[:, 0].mean():.3f}, Std: {X_scaled[:, 0].std():.3f}")
    print(f"  Feature 2 - Mean: {X_scaled[:, 1].mean():.3f}, Std: {X_scaled[:, 1].std():.3f}")
    print(f"  Range: [{X_scaled.min():.2f}, {X_scaled.max():.2f}]")
    print()

print('✅ Scaling comparison complete!')

## Step 5: Feature Engineering with PCA

Create new uncorrelated features.

In [ ]:
# Use StandardScaler + PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clean)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"=== PCA FEATURE ENGINEERING ===\n")
print(f"Explained variance ratio:")
print(f"  PC1: {pca.explained_variance_ratio_[0]:.2%}")
print(f"  PC2: {pca.explained_variance_ratio_[1]:.2%}")
print(f"  Total: {pca.explained_variance_ratio_.sum():.2%}")

print(f"\nPrincipal Components (loadings):")
print(f"  PC1: {pca.components_[0]}")
print(f"  PC2: {pca.components_[1]}")

# Visualize transformation
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(X_scaled[:, 0], X_scaled[:, 1], s=30, alpha=0.6, c='blue')
axes[0].set_xlabel('Feature 1 (scaled)', fontsize=11)
axes[0].set_ylabel('Feature 2 (scaled)', fontsize=11)
axes[0].set_title('After Scaling', fontweight='bold', fontsize=12)
axes[0].set_aspect('equal')
axes[0].grid(True, alpha=0.3)

# Add PCA vectors
for i, (comp, var) in enumerate(zip(pca.components_, pca.explained_variance_)):
    axes[0].arrow(0, 0, comp[0]*3*np.sqrt(var), comp[1]*3*np.sqrt(var),
                 head_width=0.3, head_length=0.3, fc=f'C{i}', ec=f'C{i}',
                 linewidth=2, label=f'PC{i+1}')
axes[0].legend()

axes[1].scatter(X_pca[:, 0], X_pca[:, 1], s=30, alpha=0.6, c='green')
axes[1].set_xlabel('PC1', fontsize=11)
axes[1].set_ylabel('PC2', fontsize=11)
axes[1].set_title('After PCA (uncorrelated features)', fontweight='bold', fontsize=12)
axes[1].set_aspect('equal')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('\n✅ Feature engineering complete!')

## Step 6: Complete Pipeline

In [ ]:
# Define complete preprocessing pipeline
from sklearn.pipeline import Pipeline

preprocessing_pipeline = Pipeline([
    ('outlier_detection', IsolationForest(contamination=0.1, random_state=42)),
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=0.95))  # Keep 95% variance
])

# Note: Isolation Forest doesn't have transform, so we do it manually
print("=== COMPLETE PREPROCESSING PIPELINE ===\n")

# Step 1: Outlier detection
iso = IsolationForest(contamination=0.1, random_state=42)
outliers = iso.fit_predict(X_raw)
X_step1 = X_raw[outliers == 1]
print(f"Step 1 - Outlier Detection:")
print(f"  Removed {sum(outliers == -1)} outliers")
print(f"  Remaining: {len(X_step1)} samples")

# Step 2: Scaling
scaler = StandardScaler()
X_step2 = scaler.fit_transform(X_step1)
print(f"\nStep 2 - Standardization:")
print(f"  Mean: {X_step2.mean():.2e}, Std: {X_step2.std():.2f}")

# Step 3: PCA
pca = PCA(n_components=0.95)
X_final = pca.fit_transform(X_step2)
print(f"\nStep 3 - PCA Dimensionality Reduction:")
print(f"  Original dimensions: {X_step2.shape[1]}")
print(f"  Reduced dimensions: {X_final.shape[1]}")
print(f"  Variance retained: {pca.explained_variance_ratio_.sum():.2%}")

print(f"\n✅ Pipeline complete!")
print(f"   Input: {X_raw.shape} → Output: {X_final.shape}")

## Production Best Practices

### Preprocessing Decision Tree

```
Start
  ├─ Has outliers? → YES → Use Isolation Forest or LOF
  ├─ Different scales? → YES → Use StandardScaler (or RobustScaler if outliers)
  ├─ Need [0,1] range? → YES → Use MinMaxScaler
  ├─ High dimensional? → YES → Use PCA
  └─ Ready for clustering!
```

### Scaler Comparison

| Scaler | When to Use | Pros | Cons |
|--------|------------|------|------|
| **StandardScaler** | Default choice | Mean=0, Std=1, works with most algorithms | Sensitive to outliers |
| **RobustScaler** | Data with outliers | Uses median/IQR, robust to outliers | Doesn't guarantee specific range |
| **MinMaxScaler** | Need [0,1] range | Bounded output | Very sensitive to outliers |

### Outlier Detection

- **Isolation Forest**: Fast, works in high dimensions, no assumptions
- **LOF**: Better for local outliers, slower
- **Rule of thumb**: Remove < 10% of data as outliers

### Complete Production Pipeline

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest

# 1. Detect and remove outliers
iso = IsolationForest(contamination=0.05)
outliers = iso.fit_predict(X)
X_clean = X[outliers == 1]

# 2. Build preprocessing pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=0.95))
])

X_preprocessed = pipeline.fit_transform(X_clean)

# 3. Save pipeline for production
import joblib
joblib.dump(pipeline, 'preprocessing_pipeline.pkl')

# 4. Use on new data
X_new_preprocessed = pipeline.transform(X_new)
```

### Checklist

✅ **Always visualize** before and after preprocessing  
✅ **Check for outliers** with box plots, histograms  
✅ **Standardize** for distance-based algorithms  
✅ **Keep 95% variance** when using PCA  
✅ **Save preprocessing pipeline** for consistency  
✅ **Validate** that preprocessing improves downstream task